 # Deep Learning Competition 3

 Team 13  
113062644  
113501525  
112078502  

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import math
import re
import string
import random
from PIL import Image

import tensorflow as tf
print(tf.__version__)
from tensorflow import keras, einsum
from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import Layer
from keras import ops as K
import tensorflow.keras.layers as nn

# text encoder
from tensorflow import keras
from transformers import AutoTokenizer, TFCLIPTextModel

from einops import rearrange
from einops.layers.tensorflow import Rearrange
from functools import partial
from inspect import isfunction

# Suppressing tf.hub warnings
tf.get_logger().setLevel("ERROR")

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Restrict TensorFlow to only use the first GPU
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')

        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

2025-12-08 22:44:30.816050: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-08 22:44:30.850829: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-08 22:44:31.483511: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2.20.0
1 Physical GPUs, 1 Logical GPUs


I0000 00:00:1765205072.556678 3427394 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10245 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6


# Hyperparameter

In [ ]:
# data
data_path = './dataset'
IMAGE_HEIGHT = 64
IMAGE_WIDTH = 64
IMAGE_CHANNEL = 3

BATCH_SIZE = 64

# diffusion
DIFF_STEPS = 20 # sample_steps also use the same
DIFFUSION_EPOCHS = 160

# classifier-free guidance 的 drop 機率
COND_DROP_PROB = 0.1

# inference
EMA_DECAY = 0.9995

# Text encoder and Training dataset

In [ ]:
'''Text Encoder'''
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

clip_tokenizer = AutoTokenizer.from_pretrained(CLIP_MODEL_NAME)
text_encoder = TFCLIPTextModel.from_pretrained(CLIP_MODEL_NAME)
text_encoder.trainable = False # clip is load from pretrain, only train Unet

# CLIP 的 padding token id（之後 classifier-free guidance 會用到）
pad_id = int(clip_tokenizer.pad_token_id)
MAX_CLIP_LEN = clip_tokenizer.model_max_length #max len of clip = 77

''' Training Dataset'''
dictionary_path = './dictionary'
vocab = np.load(dictionary_path + '/vocab.npy')
word2Id_dict = dict(np.load(dictionary_path + '/word2Id.npy'))
id2word_dict = dict(np.load(dictionary_path + '/id2Word.npy'))

def id_list_to_sentence(id_list):
    words = []
    for idx in id_list:
        w = id2word_dict[str(idx)]
        if w in ('<PAD>', '<RARE>'):
            continue
        words.append(w)
    return ' '.join(words)

# 2. load dataset
def training_data_generator(caption, image_path):
    # load in the image according to image path
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([None, None, 3])
    # Data Augmentation
    target_resize = 72 # resize and crop, as random crop
    img = tf.image.resize(img, size=[target_resize, target_resize])
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_crop(img, size=[IMAGE_HEIGHT, IMAGE_WIDTH, 3])

    img = tf.clip_by_value(img, 0.0, 1.0)
    img.set_shape([IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNEL])

    caption = tf.cast(caption, tf.int32)
    return img, caption

def dataset_generator(filenames, batch_size, data_generator):
    df = pd.read_pickle(filenames)
    captions_col = df['Captions'].values
    image_paths_col = df['ImagePath'].values
    print(f"Training dataset entries: {len(df)}")
    texts = []
    image_paths = []
    for img_path, caps in zip(image_paths_col, captions_col):
        for seq in caps: # take all caps per entry
            sentence = id_list_to_sentence(seq)
            if sentence.strip() == "": # skip blank sentence
                continue
            texts.append(sentence)
            image_paths.append(img_path)

    texts = np.array(texts)
    image_paths = np.array(image_paths)
    print(f"check texts item: {texts[0]}")
    print(f"check entries count: {len(texts)}, {len(image_paths)}\n")

    # CLIP tokenizer -> input_ids
    encoded = clip_tokenizer(
        list(texts),
        padding="max_length",
        max_length=MAX_CLIP_LEN,
        truncation=True,
        return_tensors="np",
    )
    clip_ids = encoded["input_ids"].astype("int32")

    assert clip_ids.shape[0] == image_paths.shape[0]

    dataset = tf.data.Dataset.from_tensor_slices((clip_ids, image_paths))
    dataset = (
        dataset
        .map(data_generator, num_parallel_calls=tf.data.AUTOTUNE)
        .cache()
        .repeat(3)
        .shuffle(10 * batch_size)
        .batch(batch_size, drop_remainder=True)
        .prefetch(buffer_size=tf.data.AUTOTUNE)
    )
    return dataset


# execute
dataset = dataset_generator(data_path + '/text2ImgData.pkl', BATCH_SIZE, training_data_generator)

# print check
for images, caption_ids in dataset.take(1):
    print(images.shape, images.dtype)
    print(caption_ids.shape, caption_ids.dtype)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at openai/clip-vit-base-patch32 were not used when initializing TFCLIPTextModel: ['clip/vision_model/encoder/layers_._11/mlp/fc2/bias:0', 'clip/vision_model/encoder/layers_._10/self_attn/q_proj/kernel:0', 'clip/vision_model/encoder/layers_._2/layer_norm2/gamma:0', 'clip/vision_model/encoder/layers_._8/self_attn/q_proj/kernel:0', 'clip/vision_model/encoder/layers_._5/mlp/fc2/kernel:0', 'clip/vision_model/encoder/layers_._4/mlp/fc1/bias:0', 'clip/vision_model/encoder/layers_._4/layer_norm2/gamma:0', 'clip/vision_model/encoder/layers_._11/self_attn/v_proj/bias:0', 'clip/vision_model/encoder/layers_._2/self_attn/k_proj/bias:0', 'clip/vision_model/encoder/layers_._6/self_attn/k_proj/kernel:0', 'clip/vision_model/encoder/layers_._10/self_attn/k_proj/kernel:0', 'clip/vision_model/encoder/layer

Training dataset entries: 7370
check texts item: the petals of the flower are pink in color and have a yellow center
check entries count: 70495, 70495

(64, 64, 64, 3) <dtype: 'float32'>
(64, 77) <dtype: 'int32'>


2025-12-08 22:54:19.841195: W tensorflow/core/kernels/data/cache_dataset_ops.cc:917] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-12-08 22:54:19.842331: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


# Model - Unet

In [ ]:

class GroupNorm(Layer):
    def __init__(self, groups=8, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.groups = groups
        self.epsilon = epsilon

    def build(self, input_shape):
        channels = int(input_shape[-1])
        if channels % self.groups != 0:
            raise ValueError(f"GroupNorm: channels ({channels}) 必須可以被 groups ({self.groups}) 整除")
        self.group_size = channels // self.groups
        self.gamma = self.add_weight(name="gamma", shape=(channels,), initializer="ones", trainable=True)
        self.beta = self.add_weight(name="beta", shape=(channels,), initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, x):
        input_shape = tf.shape(x)
        B, H, W, C = input_shape[0], input_shape[1], input_shape[2], input_shape[3]
        x = tf.reshape(x, [B, H, W, self.groups, self.group_size])
        mean, var = tf.nn.moments(x, axes=[1, 2, 4], keepdims=True)
        x = (x - mean) / tf.sqrt(var + self.epsilon)
        x = tf.reshape(x, [B, H, W, C])
        gamma = tf.reshape(self.gamma, [1, 1, 1, C])
        beta = tf.reshape(self.beta, [1, 1, 1, C])
        return x * gamma + beta

In [ ]:
# helpers functions
def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d

class SinusoidalPosEmb(Layer):
    def __init__(self, dim, max_positions=10000):
        super(SinusoidalPosEmb, self).__init__()
        self.dim = dim
        self.max_positions = max_positions

    def call(self, x, training=True):
        x = tf.cast(x, tf.float32)
        half_dim = self.dim // 2
        emb = math.log(self.max_positions) / (half_dim - 1)
        emb = tf.exp(tf.range(half_dim, dtype=tf.float32) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = tf.concat([tf.sin(emb), tf.cos(emb)], axis=-1)
        return emb

class Identity(Layer):
    def __init__(self):
        super(Identity, self).__init__()
    def call(self, x, training=True):
        return tf.identity(x)


class Residual(Layer):
    def __init__(self, fn):
        super(Residual, self).__init__()
        self.fn = fn

    def call(self, x, **kwargs):
        return self.fn(x, **kwargs) + x

def Upsample(dim):
    return nn.Conv2DTranspose(filters=dim, kernel_size=4, strides=2, padding='SAME')

def Downsample(dim):
    return nn.Conv2D(filters=dim, kernel_size=4, strides=2, padding='SAME')

class LayerNorm(Layer):
    def __init__(self, dim, eps=1e-5, **kwargs):
        super(LayerNorm, self).__init__(**kwargs)
        self.eps = eps
        self.g = tf.Variable(tf.ones([1, 1, 1, dim]))
        self.b = tf.Variable(tf.zeros([1, 1, 1, dim]))

    def call(self, x, training=True):
        var = tf.math.reduce_variance(x, axis=-1, keepdims=True)
        mean = tf.reduce_mean(x, axis=-1, keepdims=True)
        x = (x - mean) / tf.sqrt((var + self.eps)) * self.g + self.b
        return x

class PreNorm(Layer):
    def __init__(self, dim, fn):
        super(PreNorm, self).__init__()
        self.fn = fn
        self.norm = LayerNorm(dim)

    def call(self, x, **kwargs):
        x = self.norm(x)
        return self.fn(x, **kwargs)

class SiLU(Layer):
    def __init__(self):
        super(SiLU, self).__init__()
    def call(self, x, training=True):
        return x * tf.nn.sigmoid(x)

def gelu(x, approximate=False):
    if approximate:
        coeff = tf.cast(0.044715, x.dtype)
        return 0.5 * x * (1.0 + tf.tanh(0.7978845608028654 * (x + coeff * tf.pow(x, 3))))
    else:
        return 0.5 * x * (1.0 + tf.math.erf(x / tf.cast(1.4142135623730951, x.dtype)))

class GELU(Layer):
    def __init__(self, approximate=False):
        super(GELU, self).__init__()
        self.approximate = approximate
    def call(self, x, training=True):
        return gelu(x, self.approximate)


In [ ]:
# building block modules
class Block(Layer):
    def __init__(self, dim, groups=8):
        super(Block, self).__init__()
        self.proj = nn.Conv2D(dim, kernel_size=3, strides=1, padding='SAME')
        self.norm = GroupNorm(groups=groups, epsilon=1e-5)
        self.act = SiLU()


    def call(self, x, gamma_beta=None, training=True):
        x = self.proj(x)
        x = self.norm(x)

        if exists(gamma_beta):
            gamma, beta = gamma_beta
            x = x * (gamma + 1) + beta

        x = self.act(x)
        return x

class ResnetBlock(Layer):
    def __init__(self, dim, dim_out, time_emb_dim=None, groups=8):
        super(ResnetBlock, self).__init__()

        self.mlp = Sequential([
            SiLU(),
            nn.Dense(units=dim_out * 2)
        ]) if exists(time_emb_dim) else None

        self.block1 = Block(dim_out, groups=groups)
        self.block2 = Block(dim_out, groups=groups)
        self.res_conv = nn.Conv2D(filters=dim_out, kernel_size=1, strides=1) if dim != dim_out else Identity()

    def call(self, x, time_emb=None, training=True):
        gamma_beta = None
        if exists(self.mlp) and exists(time_emb):
            time_emb = self.mlp(time_emb)
            time_emb = rearrange(time_emb, 'b c -> b 1 1 c')
            gamma_beta = tf.split(time_emb, num_or_size_splits=2, axis=-1)

        h = self.block1(x, gamma_beta=gamma_beta, training=training)
        h = self.block2(h, training=training)

        return h + self.res_conv(x)

class LinearAttention(Layer):
    def __init__(self, dim, heads=4, dim_head=32):
        super(LinearAttention, self).__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        self.hidden_dim = dim_head * heads

        self.attend = nn.Softmax()
        self.to_qkv = nn.Conv2D(filters=self.hidden_dim * 3, kernel_size=1, strides=1, use_bias=False)

        self.to_out = Sequential([
            nn.Conv2D(filters=dim, kernel_size=1, strides=1),
            LayerNorm(dim)
        ])

    def call(self, x, training=True):
        b, h, w, c = x.shape
        qkv = self.to_qkv(x)
        qkv = tf.split(qkv, num_or_size_splits=3, axis=-1)
        q, k, v = map(lambda t: rearrange(t, 'b x y (h c) -> b h c (x y)', h=self.heads), qkv)

        q = tf.nn.softmax(q, axis=-2)
        k = tf.nn.softmax(k, axis=-1)

        q = q * self.scale
        context = einsum('b h d n, b h e n -> b h d e', k, v)

        out = einsum('b h d e, b h d n -> b h e n', context, q)
        out = rearrange(out, 'b h c (x y) -> b x y (h c)', h=self.heads, x=h, y=w)
        out = self.to_out(out, training=training)

        return out

class Attention(Layer):
    def __init__(self, dim, heads=4, dim_head=32):
        super(Attention, self).__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        self.hidden_dim = dim_head * heads

        self.to_qkv = nn.Conv2D(filters=self.hidden_dim * 3, kernel_size=1, strides=1, use_bias=False)
        self.to_out = nn.Conv2D(filters=dim, kernel_size=1, strides=1)

    def call(self, x, training=True):
        b, h, w, c = x.shape
        qkv = self.to_qkv(x)
        qkv = tf.split(qkv, num_or_size_splits=3, axis=-1)
        q, k, v = map(lambda t: rearrange(t, 'b x y (h c) -> b h c (x y)', h=self.heads), qkv)
        q = q * self.scale

        sim = einsum('b h d i, b h d j -> b h i j', q, k)
        sim_max = tf.stop_gradient(tf.expand_dims(tf.argmax(sim, axis=-1), axis=-1))
        sim_max = tf.cast(sim_max, tf.float32)
        sim = sim - sim_max
        attn = tf.nn.softmax(sim, axis=-1)

        out = einsum('b h i j, b h d j -> b h i d', attn, v)
        out = rearrange(out, 'b h (x y) d -> b x y (h d)', x = h, y = w)
        out = self.to_out(out, training=training)

        return out

In [ ]:
# --- [NEW] Cross Attention Layer ---
class CrossAttention(Layer):
    def __init__(self, dim, heads=4, dim_head=32):
        super(CrossAttention, self).__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        self.hidden_dim = dim_head * heads

        # Q from Image
        self.to_q = nn.Conv2D(filters=self.hidden_dim, kernel_size=1, strides=1, use_bias=False)

        # K, V from Text (Context)
        self.to_k = nn.Dense(units=self.hidden_dim, use_bias=False)
        self.to_v = nn.Dense(units=self.hidden_dim, use_bias=False)

        self.to_out = Sequential([
            nn.Conv2D(filters=dim, kernel_size=1, strides=1),
            LayerNorm(dim)
        ])

    def call(self, x, context, training=True):
        # x: [B, H, W, C]
        # context: [B, Seq_Len, C_text]
        b, h, w, c = x.shape

        q = self.to_q(x)
        k = self.to_k(context)
        v = self.to_v(context)

        # Split heads
        q = rearrange(q, 'b x y (h c) -> b h (x y) c', h=self.heads)
        k = rearrange(k, 'b s (h c) -> b h s c', h=self.heads)
        v = rearrange(v, 'b s (h c) -> b h s c', h=self.heads)

        q = q * self.scale

        # Attention scores
        sim = einsum('b h i d, b h j d -> b h i j', q, k)
        attn = tf.nn.softmax(sim, axis=-1)

        # Aggregation
        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h (x y) c -> b x y (h c)', x=h, y=w)

        return self.to_out(out, training=training) + x # Residual inside



class MLP(Layer):
    def __init__(self, hidden_dim, **kwargs):
        super(MLP, self).__init__(**kwargs)
        self.net = Sequential([
            Rearrange('... -> ... 1'),
            nn.Dense(units=hidden_dim),
            GELU(),
            LayerNorm(hidden_dim),
            nn.Dense(units=hidden_dim),
            GELU(),
            LayerNorm(hidden_dim),
            nn.Dense(units=hidden_dim),
        ])

    def call(self, x, training=True):
        return self.net(x, training=training)

In [ ]:
class Unet_conditional(Model):
    def __init__(self,
                 dim=64,
                 init_dim=None,
                 out_dim=None,
                 dim_mults=(1, 2, 4, 8),
                 channels=3,
                 resnet_block_groups=8,
                 learned_variance=False,
                 sinusoidal_cond_mlp=True,
                 in_res=64
                 ):
        super(Unet_conditional, self).__init__()

        # determine dimensions
        self.channels = channels
        self.in_res = in_res

        init_dim = default(init_dim, dim // 3 * 2)
        self.init_conv = nn.Conv2D(filters=init_dim, kernel_size=7, strides=1, padding='SAME')

        dims = [init_dim, *map(lambda m: dim * m, dim_mults)]
        in_out = list(zip(dims[:-1], dims[1:]))

        block_klass = partial(ResnetBlock, groups = resnet_block_groups)

        # time embeddings
        time_dim = dim * 4
        self.sinusoidal_cond_mlp = sinusoidal_cond_mlp

        if sinusoidal_cond_mlp:
            self.time_mlp = Sequential([
                SinusoidalPosEmb(dim),
                nn.Dense(units=time_dim),
                GELU(),
                nn.Dense(units=time_dim)
            ], name="time_embeddings")
        else:
            self.time_mlp = MLP(time_dim)

        # layers
        self.downs = []
        self.ups = []
        num_resolutions = len(in_out)

        for ind, (dim_in, dim_out) in enumerate(in_out):
            is_last = ind >= (num_resolutions - 1)

            self.downs.append([
                block_klass(dim_in, dim_out, time_emb_dim=time_dim),
                block_klass(dim_out, dim_out, time_emb_dim=time_dim),
                Residual(PreNorm(dim_out, LinearAttention(dim_out))),
                CrossAttention(dim_out),      # 這裡改掉
                Downsample(dim_out) if not is_last else Identity()
            ])

        mid_dim = dims[-1]
        self.mid_block1 = block_klass(mid_dim, mid_dim, time_emb_dim=time_dim)
        self.mid_attn = Residual(PreNorm(mid_dim, Attention(mid_dim)))
        self.mid_cross_attn = CrossAttention(mid_dim)   # 改掉
        self.mid_block2 = block_klass(mid_dim, mid_dim, time_emb_dim=time_dim)

        for ind, (dim_in, dim_out) in enumerate(reversed(in_out[1:])):
            is_last = ind >= (num_resolutions - 1)

            self.ups.append([
                block_klass(dim_out * 2, dim_in, time_emb_dim=time_dim),
                block_klass(dim_in, dim_in, time_emb_dim=time_dim),
                Residual(PreNorm(dim_in, LinearAttention(dim_in))),
                CrossAttention(dim_in),            # 改掉
                Upsample(dim_in) if not is_last else Identity()
            ])

        default_out_dim = channels * (1 if not learned_variance else 2)
        self.out_dim = default(out_dim, default_out_dim)

        self.final_conv = Sequential([
            block_klass(dim * 2, dim),
            nn.Conv2D(filters=self.out_dim, kernel_size=1, strides=1)
        ], name="output")

    def call(self, x, time=None, context=None, training=True, **kwargs):
        x = self.init_conv(x)
        t = self.time_mlp(time)

        h = []

        # Downsample Path
        for block1, block2, attn, cross_attn, downsample in self.downs:
            x = block1(x, t)
            x = block2(x, t)
            x = attn(x)                 # Linear Self-Attention
            x = cross_attn(x, context)  # Cross-Attention
            h.append(x)
            x = downsample(x)

        # Middle Path
        x = self.mid_block1(x, t)
        x = self.mid_attn(x)            # Standard Self-Attention
        x = self.mid_cross_attn(x, context)
        x = self.mid_block2(x, t)

        # Upsample Path
        for block1, block2, attn, cross_attn, upsample in self.ups:
            x = tf.concat([x, h.pop()], axis=-1)
            x = block1(x, t)
            x = block2(x, t)
            x = attn(x)                 # Linear Self-Attention
            x = cross_attn(x, context)  # Cross-Attention
            x = upsample(x)

        x = tf.concat([x, h.pop()], axis=-1)
        x = self.final_conv(x)
        return x

## init Unet and Load checkpoint

In [ ]:
unet = Unet_conditional(
    dim=64,
    dim_mults=(1, 2, 2, 4),
    in_res=IMAGE_HEIGHT,
    channels=IMAGE_CHANNEL
)

ema_unet = Unet_conditional(
    dim=64,
    dim_mults=(1, 2, 2, 4),
    in_res=IMAGE_HEIGHT,
    channels=IMAGE_CHANNEL
)


clip_hidden_dim = text_encoder.config.hidden_size
dummy_img = tf.ones([1, IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNEL])
dummy_time = tf.constant([0], dtype=tf.int32)
dummy_context = tf.ones([1, MAX_CLIP_LEN, clip_hidden_dim])
_ = unet(dummy_img, dummy_time, dummy_context)
_ = ema_unet(dummy_img, dummy_time, dummy_context)

'''Load checkpoint'''
ckpt = tf.train.Checkpoint(
    unet=unet,
    ema_unet=ema_unet,
    text_encoder=text_encoder
)
ckpt_manager = tf.train.CheckpointManager(ckpt, "./checkpoints", max_to_keep=None)

if ckpt_manager.latest_checkpoint:
    ckpt.restore(ckpt_manager.latest_checkpoint).expect_partial()
    print("Restored from {}".format(ckpt_manager.latest_checkpoint))
else:
    print("Initializing from scratch.")
    ema_unet.set_weights(unet.get_weights())

# # laod specific path
# target_ckpt = "./checkpoints_new/ckpt-50"
# ckpt.restore(target_ckpt).expect_partial()
# print("Loaded checkpoint:", target_ckpt)


/home/ttyang24/miniconda3/envs/py312/lib/python3.12/site-packages/keras/src/layers/layer.py:1493: UserWarning: Layer 'pre_norm_64' looks like it has unbuilt state, but Keras is not able to trace the layer `call()` in order to build it automatically. Possible causes:
1. The `call()` method of your layer may be crashing. Try to `__call__()` the layer eagerly on some test input first to see if it works. E.g. `x = np.random.random((3, 4)); y = layer(x)`
2. If the `call()` method is correct, then you may need to implement the `def build(self, input_shape)` method on your layer. It should create all variables used by the layer (e.g. by calling `layer.build()` on all its children layers).
Exception encountered: ''got an unexpected keyword argument 'kwargs'''
  warnings.warn(
/home/ttyang24/miniconda3/envs/py312/lib/python3.12/site-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'pre_norm_64', however the layer does not have a `build()` method implemented and

Loaded checkpoint: ./checkpoints_new/ckpt-50


# Model - Diffusion DDIM

In [ ]:
# === Loss Function ===
def diffusion_loss(true_noise, pred_noise):
    # 標準 DDPM 噪音 MSE loss (在 latent 空間)
    return tf.reduce_mean(tf.square(true_noise - pred_noise))

# === timestamp ===
min_signal_rate = 0.02
max_signal_rate = 0.95
def diffusion_schedule(diffusion_times):
    start_angle = tf.acos(max_signal_rate)
    end_angle   = tf.acos(min_signal_rate)
    diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)

    signal_rates = tf.cos(diffusion_angles)   # sqrt(alpha)
    noise_rates  = tf.sin(diffusion_angles)   # sqrt(1-alpha)
    return noise_rates, signal_rates

# === reverse_diffusion ===
def reverse_diffusion(initial_noise, context, diffusion_steps):

    num_images = tf.shape(initial_noise)[0]
    step_size  = 1.0 / float(diffusion_steps)

    next_noisy = initial_noise

    for step in range(diffusion_steps):
        step_f = tf.cast(step, tf.float32)

        diffusion_times = tf.ones((num_images, 1, 1, 1), dtype=tf.float32) - step_f * step_size
        noise_rates, signal_rates = diffusion_schedule(diffusion_times)
        t_scalar = tf.squeeze(diffusion_times, axis=[1, 2, 3])  # [B]

        pred_noise = ema_unet(next_noisy, t_scalar, context=context, training=False)
        pred_x0    = (next_noisy - noise_rates * pred_noise) / signal_rates

        next_diffusion_times = diffusion_times - step_size
        next_diffusion_times = tf.clip_by_value(next_diffusion_times, 0.0, 1.0)
        noise_rates_next, signal_rates_next = diffusion_schedule(next_diffusion_times)

        next_noisy = signal_rates_next * pred_x0 + noise_rates_next * pred_noise

    return pred_x0


def reverse_diffusion_cfg(initial_noise,
                          context_cond,
                          context_uncond,
                          diffusion_steps,
                          guidance_scale=3.0):

    num_images = tf.shape(initial_noise)[0]
    step_size  = 1.0 / float(diffusion_steps)

    next_noisy = initial_noise

    for step in range(diffusion_steps):
        step_f = tf.cast(step, tf.float32)


        diffusion_times = tf.ones((num_images, 1, 1, 1), dtype=tf.float32) - step_f * step_size
        noise_rates, signal_rates = diffusion_schedule(diffusion_times)
        t_scalar = tf.squeeze(diffusion_times, axis=[1, 2, 3])  # [B]


        eps_uncond = ema_unet(next_noisy, t_scalar, context=context_uncond, training=False)
        eps_cond   = ema_unet(next_noisy, t_scalar, context=context_cond, training=False)

        pred_noise = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        pred_x0 = (next_noisy - noise_rates * pred_noise) / signal_rates

        next_diffusion_times = diffusion_times - step_size
        next_diffusion_times = tf.clip_by_value(next_diffusion_times, 0.0, 1.0)
        noise_rates_next, signal_rates_next = diffusion_schedule(next_diffusion_times)

        next_noisy = signal_rates_next * pred_x0 + noise_rates_next * pred_noise

    return pred_x0

## Define Training step

In [ ]:

class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, warmup_steps, total_steps, min_lr=1e-6):
        super().__init__()
        self.initial_lr = initial_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        total_steps = tf.cast(self.total_steps, tf.float32)

        # Warmup phase
        warmup_lr = self.initial_lr * (step / warmup_steps)

        # Cosine decay phase
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        cosine_decay = 0.5 * (1.0 + tf.cos(np.pi * progress))
        decayed_lr = (self.initial_lr - self.min_lr) * cosine_decay + self.min_lr

        return tf.where(step < warmup_steps, warmup_lr, decayed_lr)

total_steps = DIFFUSION_EPOCHS * len(dataset)
warmup_steps = 5 * len(dataset)

lr_schedule = WarmupCosineDecay(
    initial_lr=2e-4,
    warmup_steps=2 * len(dataset),
    total_steps=total_steps,
    min_lr=1e-6
)

opt = keras.optimizers.Adam(
    learning_rate=lr_schedule,
    beta_1=0.9,
    beta_2=0.999,
    global_clipnorm=1.0,
)

@tf.function
def diffusion_train_step(images, caption_ids):
    batch_size = tf.shape(images)[0]

    diffusion_times = tf.random.uniform(
        shape=(batch_size, 1, 1, 1),
        minval=0.0,
        maxval=1.0,
        dtype=tf.float32,
    )
    noise_rates, signal_rates = diffusion_schedule(diffusion_times)
    noise   = tf.random.normal(shape=tf.shape(images))
    noisy_x = signal_rates * images + noise_rates * noise

    text_outputs = text_encoder(input_ids=caption_ids, training=False)
    text_feats   = text_outputs.last_hidden_state

    context = text_feats
    if COND_DROP_PROB > 0.0:

        uncond_ids = tf.fill(tf.shape(caption_ids), pad_id)  # [B, L]

        uncond_outputs = text_encoder(input_ids=uncond_ids, training=False)
        uncond_feats   = uncond_outputs.last_hidden_state    # [B, L, H]

        drop_mask = tf.random.uniform([batch_size, 1, 1], 0.0, 1.0) < COND_DROP_PROB
        drop_mask = tf.cast(drop_mask, tf.float32)
        context   = drop_mask * uncond_feats + (1.0 - drop_mask) * text_feats

    t_scalar = tf.squeeze(diffusion_times, axis=[1, 2, 3])  # [B]

    with tf.GradientTape() as tape:
        pred_noise = unet(noisy_x, t_scalar, context=context, training=True)
        dif_loss   = diffusion_loss(noise, pred_noise)

    vars_to_train = unet.trainable_variables
    grads = tape.gradient(dif_loss, vars_to_train)
    opt.apply_gradients(zip(grads, vars_to_train))


    for var, ema_var in zip(unet.trainable_variables, ema_unet.trainable_variables):
        ema_var.assign(EMA_DECAY * ema_var + (1.0 - EMA_DECAY) * var)

    return dif_loss

# Test Dataset

In [ ]:
def testing_data_generator(caption_ids, index):
    caption_ids = tf.cast(caption_ids, tf.int32)  # [MAX_CLIP_LEN]
    index = tf.cast(index, tf.int32)
    return caption_ids, index

def testing_dataset_generator(batch_size, data_generator):
    data = pd.read_pickle('./dataset/testData.pkl')

    raw_captions = data['Captions'].values
    print(raw_captions[0])
    ids = data['ID'].values


    texts = []
    for c in raw_captions:

        if isinstance(c, (np.ndarray, list)):
            seq = [int(x) for x in c]
        elif isinstance(c, str):
            tokens = c.replace('[', ' ').replace(']', ' ').replace(',', ' ').split()
            seq = [int(t) for t in tokens]
        else:
            seq = [int(c)]

        sentence = id_list_to_sentence(seq)
        texts.append(sentence)

    encoded = clip_tokenizer(
        texts,
        padding="max_length",
        max_length=MAX_CLIP_LEN,
        truncation=True,
        return_tensors="np",
    )
    clip_ids = encoded["input_ids"].astype("int32")
    ids_np = np.array(ids, dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((clip_ids, ids_np))
    dataset = dataset.map(data_generator, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)

    return dataset, clip_ids.shape[0]

testing_dataset, NUM_TEST = testing_dataset_generator(BATCH_SIZE, testing_data_generator)
print("Number of test captions:", NUM_TEST)


[np.str_('4'), np.str_('12'), np.str_('3'), np.str_('16'), np.str_('1'), np.str_('5'), np.str_('791'), np.str_('2'), np.str_('3'), np.str_('78'), np.str_('59'), np.str_('5427'), np.str_('5427'), np.str_('5427'), np.str_('5427'), np.str_('5427'), np.str_('5427'), np.str_('5427'), np.str_('5427'), np.str_('5427')]
Number of test captions: 819


# Define Inference (gen img)

In [ ]:
# === generate image ===
def sample_image_from_ids(caption_ids,
                          diffusion_steps=DIFF_STEPS,
                          guidance_scale=3.0):
    caption_ids = tf.convert_to_tensor(caption_ids, dtype=tf.int32)
    caption_ids = tf.reshape(caption_ids, [1, -1])   # [1, L]

    # conditional context (img align with text)
    text_out_cond = text_encoder(input_ids=caption_ids, training=False)
    context_cond  = text_out_cond.last_hidden_state  # [1, L, H]

    # uncond context to apply CFG # add
    uncond_ids = tf.fill(tf.shape(caption_ids), pad_id)     # [1, L]
    text_out_uncond = text_encoder(input_ids=uncond_ids, training=False)
    context_uncond  = text_out_uncond.last_hidden_state     # [1, L, H]

    initial_noise = tf.random.normal((1, IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNEL))

    x = reverse_diffusion_cfg(
        initial_noise,
        context_cond=context_cond,
        context_uncond=context_uncond,
        diffusion_steps=diffusion_steps,
        guidance_scale=guidance_scale,
    )
    x = tf.clip_by_value(x, 0.0, 1.0)
    return x

def run_inference_and_save(epoch, diffusion_steps=DIFF_STEPS):
    testing_dataset, NUM_TEST = testing_dataset_generator(BATCH_SIZE, testing_data_generator)
    print("Number of test captions:", NUM_TEST)

    out_dir = os.path.join("./inference", f"epoch_{epoch:03d}")
    os.makedirs(out_dir, exist_ok=True)

    saved = 0
    for caption_ids_batch, ids_batch in testing_dataset:
        for cap_ids, id_val in zip(caption_ids_batch, ids_batch):

            x = sample_image_from_ids(cap_ids, diffusion_steps=diffusion_steps)
            img = x[0].numpy()

            fname = os.path.join(out_dir, f"inference_{int(id_val):04d}.jpg")
            plt.imsave(fname, np.clip(img, 0.0, 1.0))

            saved += 1
            if saved >= NUM_TEST:
                print(f"[Inference] Epoch {epoch:03d} done, saved {saved} images to {out_dir}")
                return

    print(f"[Inference] Loop ended at epoch {epoch:03d}, saved {saved} images to {out_dir}")

# Run Training, and testing

In [ ]:
# DIFFUSION_EPOCHS = 600
for e in range(1, DIFFUSION_EPOCHS + 1):
    bar = tf.keras.utils.Progbar(len(dataset) - 1)
    dif_losses = []

    for i, (images, caption_ids) in enumerate(dataset):
        dif_loss = diffusion_train_step(images, caption_ids)
        dif_losses.append(dif_loss.numpy())
        bar.update(i, values=[("dif", dif_loss)])

    print(f"[Diff] Epoch {e}/{DIFFUSION_EPOCHS} | dif={np.mean(dif_losses):.4f}")

    if e % 10 == 0 and e >= 40:
        print(f"\n[Inference] Running test-time inference at epoch {e} ...")
        run_inference_and_save(epoch=e, diffusion_steps= DIFF_STEPS)
        ckpt_manager.save(checkpoint_number=e)


3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1254s 372ms/step - dif: 0.0279
[Diff] Epoch 1/160 | dif=0.3647


2025-12-05 02:21:33.960814: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1229s 372ms/step - dif: 0.0227
[Diff] Epoch 2/160 | dif=0.0241
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1228s 372ms/step - dif: 0.0214
[Diff] Epoch 3/160 | dif=0.0195


2025-12-05 03:02:31.851061: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1230s 372ms/step - dif: 0.0252
[Diff] Epoch 4/160 | dif=0.0181
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1228s 372ms/step - dif: 0.0211
[Diff] Epoch 5/160 | dif=0.0176
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1227s 371ms/step - dif: 0.0184
[Diff] Epoch 6/160 | dif=0.0172
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1230s 372ms/step - dif: 0.0182
[Diff] Epoch 7/160 | dif=0.0170


...

2025-12-05 07:56:43.990255: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1227s 371ms/step - dif: 0.0150
[Diff] Epoch 16/160 | dif=0.0162
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1229s 372ms/step - dif: 0.0191
[Diff] Epoch 17/160 | dif=0.0160
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1229s 372ms/step - dif: 0.0180
[Diff] Epoch 18/160 | dif=0.0160
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1227s 371ms/step - dif: 0.0175
[Diff] Epoch 19/160 | dif=0.0160
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1229s 372ms/step - dif: 0.0169
[Diff] Epoch 20/160 | dif=0.0158

[Inference] Running test-time inference at epoch 20 ...
Number of test captions: 819
[Inference] Epoch 020 done, saved 819 images to ./inference/epoch_020
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1230s 372ms/step - dif: 0.0165
[Diff] Epoch 21/160 | dif=0.0159
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 1231s 373ms/step - dif: 0.0147
[Diff] Epoch 22/160 | dif=0.0158
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 700s 212ms/step - dif: 0.0155
[Diff] Epoch 23/160 | dif=0.0157
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 669s 202ms/step - dif: 0.0191
[Diff] Epoch 24/160 | di

2025-12-05 13:38:20.523851: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


3303/3303 ━━━━━━━━━━━━━━━━━━━━ 669s 203ms/step - dif: 0.0156
[Diff] Epoch 32/160 | dif=0.0152
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 669s 203ms/step - dif: 0.0167
[Diff] Epoch 33/160 | dif=0.0152
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 669s 203ms/step - dif: 0.0171
[Diff] Epoch 34/160 | dif=0.0152
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 670s 203ms/step - dif: 0.0172
[Diff] Epoch 35/160 | dif=0.0152
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 669s 203ms/step - dif: 0.0134
[Diff] Epoch 36/160 | dif=0.0151
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 670s 203ms/step - dif: 0.0143
[Diff] Epoch 37/160 | dif=0.0151
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 669s 203ms/step - dif: 0.0170
[Diff] Epoch 38/160 | dif=0.0150
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 669s 203ms/step - dif: 0.0184
[Diff] Epoch 39/160 | dif=0.0150
3303/3303 ━━━━━━━━━━━━━━━━━━━━ 669s 203ms/step - dif: 0.0190
[Diff] Epoch 40/160 | dif=0.0150

[Inference] Running test-time inference at epoch 40 ...
Number of test captions: 819
[Inference] Epoch 040 done, saved 819 images to ./inference/epoch_04

KeyboardInterrupt: 